In [1]:
## Create a vector of the required packages for this analysis
req_packages <- c("Biostrings", "BSgenome", "GenomicFeatures", "tidyverse")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

### Set up data

In [2]:
## load in data
logfc <- read_csv("logfc_psesrmt_normalized.csv")

### create database for genes from GFF
dpse_ranges <- makeTxDbFromGFF("/home/labshare/data/genomes/obscura_group/pseudoobscura/GCF_009870125.1/genomic.gff", format = "gff")

### load in fasta of pseudoobscura
dpse_fasta <- readDNAStringSet("/home/labshare/data/genomes/obscura_group/pseudoobscura/GCF_009870125.1/GCF_009870125.1_UCI_Dpse_MV25_genomic.fna")

New names:
• `` -> `...1`
Rows: 28593 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): gene, comparison, significance, direction, color
dbl (3): ...1, logfc, pvalue

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message in call_fun_in_txdbmaker("makeTxDbFromGFF", ...):
“makeTxDbFromGFF() has moved to the txdbmaker package. Please call
  txdbmaker::makeTxDbFromGFF() to get rid of this warning.”
Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .extract_transcripts_from_GRanges(tx_IDX, gr, mcols0$type, mcols0$ID, :
“some transcripts have no "transcript_id" attribute ==> their name
  ("tx_name" column in the TxDb object) was set to NA”
Warning message in .extract_transcripts_from_GRanges(tx_IDX, gr, mcols0

In [3]:
## clean up fasta names
fasta_names <- names(dpse_fasta) %>%
    as.data.frame() %>%
    rename(original = 1) %>%
    separate_wider_delim(original, delim = ",", names = c("specifics", "scaffold", "sequencing"), cols_remove = FALSE) %>%
    select(-c(scaffold, sequencing)) %>%
    separate_wider_delim(specifics, delim = " ", names = c("name", "genus", "species", "trash1", "strain", "trash2", "chromosome"), 
                         too_many = "merge", too_few = "align_start") %>%
    select(-c(genus, species, trash1, strain)) %>%
    mutate(chromosome = case_when(trash2 == "chromosome" ~ chromosome,
                                  trash2 == "mitochondrion" ~ "mitochondrion",
                                  TRUE ~ "scaffold")) %>%
    select(original, name, chromosome)

chromosome_names <- fasta_names %>%
    pull("name")

## rename fasta chromosomes
names(dpse_fasta) <- chromosome_names

In [4]:
## create GRanges object of all of the genes
genes <- genes(dpse_ranges)

In [5]:
## create function to pull the sequences for the promoters

create_fasta <- function(df, chr) {
    ## create empty output and names
    output <- DNAStringSet()
    gene_names <- list()

    ## pull list of genes
    genes <- df$gene_id

    for (i in genes) {
        ## filter the table for just the relevent gene
        promoter <- df %>%
            filter(gene_id == i)
        
        ## get all of the sequence information
        start <- promoter %>%
            pull("start")
        end <- promoter %>%
            pull("end")
        strand <- promoter %>%
            pull("strand")
            ## get chromosome information, if not supplied
            if(missing(chr)) {
                chr_name <- promoter %>%
                    pull("seqnames")
            } else {
                chr_name <- chr
            }

        ## get the chromosome sequence
        chr_seq <- dpse_fasta[[chr_name]]

        ## only use promoter regions with valid positions
        if (between(start, 0, length(chr_seq)) & between(end, 0, length(chr_seq))) {
            
            ## add gene name to list of used genes
            gene_names <- c(gene_names, i)

            ## pull the promoter sequence
            if (strand == "+") {
                seq <- subseq(chr_seq, start = start, end = end)
            } else {
                seq_comp <- subseq(chr_seq, start = start, end = end)
                seq <- reverseComplement(seq_comp)
            }

        ##convert DNAString object into DNAStringSet
        seq_set <- DNAStringSet(seq)

        ## store it in fasta
        output <- c(output, seq_set)

        }
    }

    names(output) <- gene_names
    output
}

### Get Promoters fo All Differentially Expressed Genes

In [17]:
## create GRanges object for each category of the genes
diff_ranges <- genes %>%
  as_tibble() %>%
  filter(gene_id %in% diff_genes) %>%
  GRanges()
nondiff_ranges <- genes %>%
  as_tibble() %>%
  filter(!(gene_id %in% diff_genes)) %>%
  GRanges()
  
## get promoters of each gene category
diff_promoters <- promoters(diff_ranges, upstream = 1000) %>%
  as_tibble()
nondiff_promoters <- promoters(nondiff_ranges, upstream = 1000) %>%
  as_tibble()

## create fasta object for each category
diff_fasta <- create_fasta(diff_promoters)
nondiff_fasta <- create_fasta(nondiff_promoters)

In [18]:
## write fasta for promoters
writeXStringSet(diff_fasta, "promoter_fasta/alldifferential_promoters.fasta")
writeXStringSet(nondiff_fasta, "promoter_fasta/nondifferential_promoters.fasta")

### Get Promoters for All Upregulated and Downregulated Genes

In [19]:
## create GRanges object for each category of the genes
upregulated_ranges <- genes %>%
  as_tibble() %>%
  filter(gene_id %in% upregulated_genes) %>%
  GRanges()
downregulated_ranges <- genes %>%
  as_tibble() %>%
  filter(gene_id %in% downregulated_genes) %>%
  GRanges()
other_ranges <- genes %>%
  as_tibble() %>%
  filter(!(gene_id %in% upregulated_genes) | !(gene_id %in% downregulated_genes)) %>%
  GRanges()

## get promoters of each gene category
upregulated_promoters <- promoters(upregulated_ranges, upstream = 1000) %>%
  as_tibble()
downregulated_promoters <- promoters(downregulated_ranges, upstream = 1000) %>%
  as_tibble()
other_promoters <- promoters(other_ranges, upstream = 1000) %>%
  as_tibble()

## create fasta object for each category
upregulated_fasta <- create_fasta(upregulated_promoters)
downregulated_fasta <- create_fasta(downregulated_promoters)
other_fasta <- create_fasta(other_promoters)

In [20]:
## write fasta for promoters
writeXStringSet(upregulated_fasta, "promoter_fasta/upregulated_promoters.fasta")
writeXStringSet(downregulated_fasta, "promoter_fasta/downregulated_promoters.fasta")
writeXStringSet(other_fasta, "promoter_fasta/other_promoters.fasta")

### Get Up and Down Regulated Promoters on X Chromosome

In [21]:
## get name of x chromosome
x_name <- fasta_names %>%
    filter(chromosome == "X") %>%
    pull("name")
x_name

[1] "NC_046683.1"

In [22]:
## get list of differentially regulated promoters on x
upregulatedx_promoters <- upregulated_promoters %>%
    filter(seqnames == x_name)
downregulatedx_promoters <- downregulated_promoters %>%
    filter(seqnames == x_name)
otherx_promoters <- other_promoters %>%
    filter(seqnames == x_name)

## create fasta object
upregulatedx_fasta <- create_fasta(upregulatedx_promoters)
downregulatedx_fasta <- create_fasta(downregulatedx_promoters)
otherx_fasta <- create_fasta(otherx_promoters)

In [23]:
## write fasta for promoters
writeXStringSet(upregulatedx_fasta, "promoter_fasta/xchr_upregulated_promoters.fasta")
writeXStringSet(downregulatedx_fasta, "promoter_fasta/xchr_downregulated_promoters.fasta")
writeXStringSet(otherx_fasta, "promoter_fasta/xchr_other_promoters.fasta")

### Get Differential Promoters on X Chromosome

In [24]:
## combine the upregulated and downregulated promoters to get all differentially expressed on X
diffx_promoters <- rbind(upregulatedx_promoters, downregulatedx_promoters)

## create fasta object
diffx_fasta <- create_fasta(diffx_promoters)

In [25]:
## write fasta
writeXStringSet(diffx_fasta, "promoter_fasta/xchr_alldifferential_promoters.fasta")

### Get promoters for Testes-Specific Genes

In [26]:
## load in tau data frame
tau <- read_csv("/home/clavery/projects/pse_ag/analysis/tau/tau_tissue_0.50.csv")

## get testes specific genes
testes_genes <- tau %>%
    filter(tau_tissue == "male_gonad") %>%
    pull("gene")

Rows: 16624 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): gene, max_organ, tau_tissue
dbl (2): tau, perc

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [27]:
## get differentially expressed testes genes
upregulated_testes_promoters <- upregulated_promoters %>%
    filter(gene_id %in% testes_genes)
downregulated_testes_promoters <- downregulated_promoters %>%
    filter(gene_id %in% testes_genes)
other_testes_promoters <- other_promoters %>%
    filter(gene_id %in% testes_genes)

## create fasta object
upregulated_testes_fasta <- create_fasta(upregulated_testes_promoters)
downregulated_testes_fasta <- create_fasta(downregulated_testes_promoters)
other_testes_fasta <- create_fasta(other_testes_promoters)

## write fasta for promoters
writeXStringSet(upregulated_testes_fasta, "promoter_fasta/testes_upregulated_promoters.fasta")
writeXStringSet(downregulated_testes_fasta, "promoter_fasta/testes_downregulated_promoters.fasta")
writeXStringSet(other_testes_fasta, "promoter_fasta/testes_other_promoters.fasta")

### Get Promoters for X chromosome specific testes genes

In [28]:
## get differentially expressed testes genes
upregulatedx_testes_promoters <- upregulated_promoters %>%
    filter(gene_id %in% testes_genes) %>%
    filter(seqnames == x_name)
downregulatedx_testes_promoters <- downregulated_promoters %>%
    filter(gene_id %in% testes_genes) %>%
    filter(seqnames == x_name)
otherx_testes_promoters <- other_promoters %>%
    filter(gene_id %in% testes_genes) %>%
    filter(seqnames == x_name)

## create fasta object
upregulated_testes_fasta <- create_fasta(upregulatedx_testes_promoters)
downregulated_testes_fasta <- create_fasta(downregulatedx_testes_promoters)
other_testes_fasta <- create_fasta(otherx_testes_promoters)

## write fasta for promoters
writeXStringSet(upregulated_testes_fasta, "promoter_fasta/testes_upregulated_xchr_promoters.fasta")
writeXStringSet(downregulated_testes_fasta, "promoter_fasta/testes_downregulated_xchr_promoters.fasta")
writeXStringSet(other_testes_fasta, "promoter_fasta/testes_other_xchr_promoters.fasta")

In [29]:
## find genes that are on x and testes specific
downxtest_genes <- downregulatedx_testes_promoters %>%
    pull("gene_id")
upxtest_genes <- upregulatedx_testes_promoters %>%
    pull("gene_id")
diffxtest_genes <- c(downxtest_genes, upxtest_genes)

### Look at genes adaptive for parasperm content

In [32]:
## load in evogenex results
evogenex <- read_csv("/home/clavery/projects/pse_ag/analysis/polyspermy/intermediate/evogenex_results.csv") %>%
    na.omit()

Rows: 16881 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): gene, constraint, adaptive
dbl (6): constraint_pvalue, constraint_qvalue, adaptive_pvalue, adaptive_qva...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [33]:
## get a list of adaptive genes
adaptive_genes <- evogenex %>%
    filter(constraint == "constraint" & adaptive == "adaptive") %>%
    pull("gene")

## get promoters for adaptive genes
adaptive_ranges <- genes %>%
  as_tibble() %>%
  filter(gene_id %in% adaptive_genes) %>%
  GRanges()
nonadaptive_ranges <- genes %>%
  as_tibble() %>%
  filter(!(gene_id %in% adaptive_genes)) %>%
  GRanges()
  
## get promoters of each gene category
adaptive_promoters <- promoters(adaptive_ranges, upstream = 1000) %>%
  as_tibble()
nonadaptive_promoters <- promoters(nonadaptive_ranges, upstream = 1000) %>%
  as_tibble()

## create fasta object
adaptive_fasta <- create_fasta(adaptive_promoters)
nonadaptive_fasta <- create_fasta(nonadaptive_promoters)

## write fasta for promoters
writeXStringSet(adaptive_fasta, "promoter_fasta/adaptive_promoters.fasta")
writeXStringSet(nonadaptive_fasta, "promoter_fasta/nonadaptive_promoters.fasta")

### Look at genes associated with parasperm branches

In [34]:
## load in credible gene data frame
credible_df <- read_csv("/home/clavery/projects/pse_ag/analysis/polyspermy/intermediate/extreme_credible_genes.csv")

## get genes with credible changes at nodes with increases in gene expression (without conflicting if present in multiple)
credible_genes <- credible_df %>%
    filter(!is.na(x7)) %>%
    pull("transcript_id")

Rows: 873 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): transcript_id, x7, x12, x14, x15
num (1): signif

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [35]:
pos_credible_genes <- credible_df %>%
    filter(!is.na(x7)) %>%
    select(-signif) %>%
    pivot_longer(!transcript_id, names_to = "trash", values_to = "value") %>%
    filter(grepl("\\*", value)) %>%
    mutate(value = str_replace_all(value, "\\*", ""),
           value = as.double(value)) %>%
    filter(value > 0) %>%
    pull(transcript_id) %>%
    unique()
length(pos_credible_genes)

[1] 212

In [36]:
## get promoters for credible genes
credible_ranges <- genes %>%
  as_tibble() %>%
  filter(gene_id %in% credible_genes) %>%
  GRanges()
pos_credible_ranges <- genes %>%
  as_tibble() %>%
  filter(gene_id %in% pos_credible_genes) %>%
  GRanges()
noncredible_ranges <- genes %>%
  as_tibble() %>%
  filter(!(gene_id %in% credible_genes)) %>%
  GRanges()
  
## get promoters of each gene category
credible_promoters <- promoters(credible_ranges, upstream = 1000) %>%
  as_tibble()
pos_credible_promoters <- promoters(pos_credible_ranges, upstream = 1000) %>%
  as_tibble()
noncredible_promoters <- promoters(noncredible_ranges, upstream = 1000) %>%
  as_tibble()

## create fasta object
credible_fasta <- create_fasta(credible_promoters)
pos_credible_fasta <- create_fasta(pos_credible_promoters)
noncredible_fasta <- create_fasta(noncredible_promoters)

## write fasta for promoters
writeXStringSet(credible_fasta, "promoter_fasta/credible_promoters.fasta")
writeXStringSet(pos_credible_fasta, "promoter_fasta/pos_credible_promoters.fasta")
writeXStringSet(noncredible_fasta, "promoter_fasta/noncredible_promoters.fasta")